## Student Attendance Analytics with Pandas

**Dataset:** `aiml_attendance.xlsx` (the provided workbook). The loader also accepts `attendance.xlsx`.

**How to run:** Keep this notebook and the Excel workbook in the same folder, then use **Run All** from the top.

**Note:** Q19 intentionally creates a *hypothetical* Student_Details table because the supplied workbook contains attendance information but no separate Branch/Semester table. Those details are illustrative, not verified student records.

In [1]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [2]:
# Keep aiml_attendance.xlsx (your uploaded dataset) in the same folder as this notebook.
# The cell also accepts attendance.xlsx if you rename the workbook.
from pathlib import Path

possible_files = [Path("aiml_attendance.xlsx"), Path("attendance.xlsx")]
data_file = next((file for file in possible_files if file.exists()), None)
if data_file is None:
    raise FileNotFoundError(
        "Could not find aiml_attendance.xlsx or attendance.xlsx. "
        "Keep your Excel dataset in the same folder as this notebook."
    )

df = pd.read_excel(data_file)
print(f"Loaded dataset: {data_file} | Shape: {df.shape}")
display(df.head())

Loaded dataset: aiml_attendance.xlsx | Shape: (59, 12)


,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW24AI001,ABHIJITH,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67%,Critical,4 more,2026-10-09,1
1,4MW24AI002,ABHISHEK KRISHNA P,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67%,Critical,4 more,2026-10-09,1
2,4MW24AI003,ADITYA,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8%,Critical,32 more,2026-09-22,5
3,4MW24AI004,AKSHATA JAKAREDDY,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92%,Safe,✓ Met,2026-10-10,0
4,4MW24AI005,ANANYA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83%,Warning,✓ Met,2026-10-09,1


#### Section 1: Data Inspection

In [3]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""

# Q1: Preview, dimensions, and data types
display(df.head(10))
print("DataFrame shape:", df.shape)
print("\nColumn data types:")
print(df.dtypes)

,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW24AI001,ABHIJITH,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67%,Critical,4 more,2026-10-09,1
1,4MW24AI002,ABHISHEK KRISHNA P,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67%,Critical,4 more,2026-10-09,1
2,4MW24AI003,ADITYA,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8%,Critical,32 more,2026-09-22,5
3,4MW24AI004,AKSHATA JAKAREDDY,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92%,Safe,✓ Met,2026-10-10,0
4,4MW24AI005,ANANYA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83%,Warning,✓ Met,2026-10-09,1
5,4MW24AI006,ANANYASHREE,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83%,Warning,✓ Met,2026-10-09,1
6,4MW24AI007,ANKITA TARANATH MESTA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83%,Warning,✓ Met,2026-10-09,1
7,4MW24AI008,ANVITH S SHERIGAR,Data Science & GenAI - 3nd Yr- AIML,12,12,0,0,100%,Safe,✓ Met,2026-10-10,0
8,4MW24AI009,APEKSHA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83%,Warning,✓ Met,2026-10-09,1
9,4MW24AI010,ARLENE VELORA FERNANDES,Data Science & GenAI - 3nd Yr- AIML,12,9,3,0,75%,Warning,✓ Met,2026-10-09,1


DataFrame shape: (59, 12)

Column data types:
Roll No                 object
Student Name            object
Subject                 object
Total Classes            int64
Present                  int64
Absent                   int64
Excused                  int64
Attendance %            object
Status                  object
Classes Needed (75%)    object
Last Present Date       object
Consecutive Absences     int64
dtype: object


In [4]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """

# Q2: Missing-value counts and "Never" values
print("Missing values in each column:")
display(df.isna().sum().to_frame("missing_count"))

# "Never" is text, so count it separately before converting the date column.
never_counts = df.astype("string").apply(
    lambda col: col.str.strip().str.casefold().eq("never").sum()
)
never_counts = never_counts[never_counts > 0]
print('Count of "Never" values by column:')
display(never_counts.to_frame("never_count") if not never_counts.empty
        else pd.DataFrame({"message": ['No "Never" values found']}))

Missing values in each column:


,missing_count
Roll No,0
Student Name,0
Subject,0
Total Classes,0
Present,0
Absent,0
Excused,0
Attendance %,0
Status,0
Classes Needed (75%),0


Count of "Never" values by column:


,never_count
Last Present Date,2


In [5]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""

# Q3: Rename columns to snake_case
import re

def to_snake_case(column):
    column = str(column).strip()
    column = column.replace("%", " pct ")
    column = re.sub(r"[^A-Za-z0-9]+", "_", column)
    return re.sub(r"_+", "_", column).strip("_").lower()

df.columns = [to_snake_case(col) for col in df.columns]
print("Updated column names:")
print(df.columns.tolist())

Updated column names:
['roll_no', 'student_name', 'subject', 'total_classes', 'present', 'absent', 'excused', 'attendance_pct', 'status', 'classes_needed_75_pct', 'last_present_date', 'consecutive_absences']


In [6]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""

# Q4: Convert attendance to numeric percentage points (e.g. "64%" -> 64.0)
# Convert the last-present-date column to datetime; "Never" becomes NaT.
if "attendance_pct" in df.columns:
    df["attendance_pct"] = pd.to_numeric(
        df["attendance_pct"].astype("string").str.replace("%", "", regex=False).str.strip(),
        errors="coerce"
    ).astype(float)
elif "attendance" in df.columns:
    df["attendance"] = pd.to_numeric(
        df["attendance"].astype("string").str.replace("%", "", regex=False).str.strip(),
        errors="coerce"
    ).astype(float)

date_candidates = [c for c in df.columns if "last_present_date" in c or c == "last_present"]
if date_candidates:
    date_col = date_candidates[0]
    df[date_col] = pd.to_datetime(
        df[date_col].replace(r"(?i)^\s*never\s*$", pd.NA, regex=True),
        errors="coerce"
    )

print("Updated data types:")
print(df.dtypes)
display(df.head())

Updated data types:
roll_no                          object
student_name                     object
subject                          object
total_classes                     int64
present                           int64
absent                            int64
excused                           int64
attendance_pct                  float64
status                           object
classes_needed_75_pct            object
last_present_date        datetime64[ns]
consecutive_absences              int64
dtype: object


,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75_pct,last_present_date,consecutive_absences
0,4MW24AI001,ABHIJITH,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67.0,Critical,4 more,2026-10-09,1
1,4MW24AI002,ABHISHEK KRISHNA P,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67.0,Critical,4 more,2026-10-09,1
2,4MW24AI003,ADITYA,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-09-22,5
3,4MW24AI004,AKSHATA JAKAREDDY,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-10,0
4,4MW24AI005,ANANYA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83.0,Warning,✓ Met,2026-10-09,1


#### Section 2: Filtering & Selection

In [7]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""

# Q5: Critical students, lowest attendance first
attendance_col = "attendance_pct" if "attendance_pct" in df.columns else "attendance"
critical_students = df[df["status"].astype("string").str.strip().str.casefold() == "critical"]
display(critical_students.sort_values(attendance_col, ascending=True))

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75_pct,last_present_date,consecutive_absences
32,4MW24AI035,RAJATH K K,Data Science & GenAI - 3nd Yr- AIML,12,0,12,0,0.0,Critical,36 more,NaT,12
33,4MW24AI036,RAJU ASHOKA RATHOD,Data Science & GenAI - 3nd Yr- AIML,12,0,12,0,0.0,Critical,36 more,NaT,12
2,4MW24AI003,ADITYA,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-09-22,5
35,4MW24AI038,RASHMI,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-10-06,2
58,4MW25AI401,Omkar Rane,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-09-08,9
22,4MW24AI025,MANMITHA,Data Science & GenAI - 3nd Yr- AIML,12,2,10,0,17.0,Critical,28 more,2026-10-06,2
41,4MW24AI045,SHRAVYA,Data Science & GenAI - 3nd Yr- AIML,12,2,10,0,17.0,Critical,28 more,2026-09-08,9
44,4MW24AI048,SINCHANA (SUNDARA MADIVALA),Data Science & GenAI - 3nd Yr- AIML,12,2,10,0,17.0,Critical,28 more,2026-10-06,2
30,4MW24AI033,PRAMOD ACHAR,Data Science & GenAI - 3nd Yr- AIML,12,3,9,0,25.0,Critical,24 more,2026-10-06,2
37,4MW24AI041,SANIKA,Data Science & GenAI - 3nd Yr- AIML,12,3,9,0,25.0,Critical,24 more,2026-10-06,2


In [8]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""

# Q6: Students with zero present days
present_col = "present" if "present" in df.columns else "present_days"
roll_col = "roll_no" if "roll_no" in df.columns else "roll_number"
name_col = "student_name" if "student_name" in df.columns else "name"
consecutive_col = "consecutive_absences"
zero_present = df[pd.to_numeric(df[present_col], errors="coerce").fillna(-1).eq(0)]
display(zero_present[[c for c in [roll_col, name_col, consecutive_col] if c in df.columns]])

,roll_no,student_name,consecutive_absences
32,4MW24AI035,RAJATH K K,12
33,4MW24AI036,RAJU ASHOKA RATHOD,12


In [9]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """

# Q7(a): Safe students using .loc
safe_students = df.loc[df["status"].astype("string").str.strip().str.casefold() == "safe"]
display(safe_students)

# Q7(b): First 15 rows and columns 2–6 (1-based column numbering)
display(df.iloc[:15, 1:6])

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75_pct,last_present_date,consecutive_absences
3,4MW24AI004,AKSHATA JAKAREDDY,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-10,0
7,4MW24AI008,ANVITH S SHERIGAR,Data Science & GenAI - 3nd Yr- AIML,12,12,0,0,100.0,Safe,✓ Met,2026-10-10,0
14,4MW24AI016,G BHUPATHE,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-10,0
15,4MW24AI018,HARSHAL G POOJARY,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-09,1
16,4MW24AI019,JNANESH . S,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-09,1
17,4MW24AI020,KEERTHAN KRISHNA SHERIGAR,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-09,1
21,4MW24AI024,MANEESH SHETTY,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-10,0
26,4MW24AI029,MUHAMMED NASHIT,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-10,0
29,4MW24AI032,PRAGNYA R,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-10,0
38,4MW24AI042,SHAMANTH K,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-09,1


,student_name,subject,total_classes,present,absent
0,ABHIJITH,Data Science & GenAI - 3nd Yr- AIML,12,8,4
1,ABHISHEK KRISHNA P,Data Science & GenAI - 3nd Yr- AIML,12,8,4
2,ADITYA,Data Science & GenAI - 3nd Yr- AIML,12,1,11
3,AKSHATA JAKAREDDY,Data Science & GenAI - 3nd Yr- AIML,12,11,1
4,ANANYA,Data Science & GenAI - 3nd Yr- AIML,12,10,2
5,ANANYASHREE,Data Science & GenAI - 3nd Yr- AIML,12,10,2
6,ANKITA TARANATH MESTA,Data Science & GenAI - 3nd Yr- AIML,12,10,2
7,ANVITH S SHERIGAR,Data Science & GenAI - 3nd Yr- AIML,12,12,0
8,APEKSHA,Data Science & GenAI - 3nd Yr- AIML,12,10,2
9,ARLENE VELORA FERNANDES,Data Science & GenAI - 3nd Yr- AIML,12,9,3


In [10]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""

# Q8: At least 5 consecutive absences
date_col = next((c for c in df.columns if "last_present_date" in c or c == "last_present"), None)
cols = [c for c in [name_col, "status", date_col] if c is not None and c in df.columns]
display(df.loc[pd.to_numeric(df[consecutive_col], errors="coerce").ge(5), cols])

,student_name,status,last_present_date
2,ADITYA,Critical,2026-09-22
25,MOURYA RAJ H N,Critical,2026-09-22
32,RAJATH K K,Critical,NaT
33,RAJU ASHOKA RATHOD,Critical,NaT
41,SHRAVYA,Critical,2026-09-08
58,Omkar Rane,Critical,2026-09-08


### Section 3: Grouping & Aggregation 

In [11]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""

# Q9: Aggregate student count, average attendance, and maximum consecutive absences by status
summary_by_status = df.groupby("status", dropna=False).agg(
    student_count=(name_col, "size"),
    average_attendance=(attendance_col, "mean"),
    max_consecutive_absences=(consecutive_col, "max")
).reset_index()
display(summary_by_status)

,status,student_count,average_attendance,max_consecutive_absences
0,Critical,23,30.086957,12
1,Safe,13,93.230769,1
2,Warning,23,80.565217,2


In [12]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""

# Q10: Categorize attendance and count students in each category
df["attendance_category"] = pd.cut(
    df[attendance_col],
    bins=[-float("inf"), 60, 75, 90, float("inf")],
    labels=["Poor", "Average", "Good", "Excellent"],
    right=False
)
# Align the category thresholds with the stated bands: <60, 60–<75, 75–<90, >=90.
display(df.groupby("attendance_category", observed=False).size().rename("student_count").reset_index())

,attendance_category,student_count
0,Poor,20
1,Average,3
2,Good,23
3,Excellent,13


In [13]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""

# Q11: Total classes conducted and total present/absent counts using .agg()
total_columns = {}
if "total_classes" in df.columns:
    total_columns["total_classes"] = "sum"
if "present" in df.columns:
    total_columns["present"] = "sum"
elif "present_days" in df.columns:
    total_columns["present_days"] = "sum"
if "absent" in df.columns:
    total_columns["absent"] = "sum"

display(df.agg(total_columns).to_frame("total").T if total_columns else
        pd.DataFrame({"message": ["Expected total_classes/present/absent columns were not found"]}))

,total_classes,present,absent
total,708,451,257


In [14]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""

# Q12: Group by status; min/max/mean present and sum absent
present_col = "present" if "present" in df.columns else "present_days"
agg_spec = {present_col: ["min", "max", "mean"]}
if "absent" in df.columns:
    agg_spec["absent"] = "sum"
display(df.groupby("status", dropna=False).agg(agg_spec))

present                absent
             min max       mean    sum
status                                
Critical       0   8   3.608696    193
Safe          11  12  11.153846     11
Warning        9  10   9.695652     53

### Section 4: Feature Engineering & Transformation

In [15]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""

# Q13: Calculate risk score and show the 10 highest-risk students
df["risk_score"] = (
    pd.to_numeric(df["absent"], errors="coerce").fillna(0) * 0.4
    + pd.to_numeric(df[consecutive_col], errors="coerce").fillna(0) * 0.6
)
display(df.sort_values("risk_score", ascending=False).head(10))

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75_pct,last_present_date,consecutive_absences,attendance_category,risk_score
33,4MW24AI036,RAJU ASHOKA RATHOD,Data Science & GenAI - 3nd Yr- AIML,12,0,12,0,0.0,Critical,36 more,NaT,12,Poor,12.0
32,4MW24AI035,RAJATH K K,Data Science & GenAI - 3nd Yr- AIML,12,0,12,0,0.0,Critical,36 more,NaT,12,Poor,12.0
58,4MW25AI401,Omkar Rane,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-09-08,9,Poor,9.8
41,4MW24AI045,SHRAVYA,Data Science & GenAI - 3nd Yr- AIML,12,2,10,0,17.0,Critical,28 more,2026-09-08,9,Poor,9.4
2,4MW24AI003,ADITYA,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-09-22,5,Poor,7.4
25,4MW24AI028,MOURYA RAJ H N,Data Science & GenAI - 3nd Yr- AIML,12,3,9,0,25.0,Critical,24 more,2026-09-22,5,Poor,6.6
35,4MW24AI038,RASHMI,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-10-06,2,Poor,5.6
18,4MW24AI021,KISHAN J,Data Science & GenAI - 3nd Yr- AIML,12,3,9,0,25.0,Critical,24 more,2026-09-29,3,Poor,5.4
22,4MW24AI025,MANMITHA,Data Science & GenAI - 3nd Yr- AIML,12,2,10,0,17.0,Critical,28 more,2026-10-06,2,Poor,5.2
44,4MW24AI048,SINCHANA (SUNDARA MADIVALA),Data Science & GenAI - 3nd Yr- AIML,12,2,10,0,17.0,Critical,28 more,2026-10-06,2,Poor,5.2


In [16]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""

# Q14: Extract numeric value from the Classes Needed (75%) column
classes_needed_col = next(
    (c for c in df.columns if "classes_needed" in c),
    None
)
if classes_needed_col is None:
    print("Column containing 'Classes Needed' was not found.")
else:
    df["classes_needed_numeric"] = (
        df[classes_needed_col].astype("string")
        .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
        .pipe(pd.to_numeric, errors="coerce")
    )
    df.loc[df[classes_needed_col].astype("string").str.contains(r"✓\s*Met|\bmet\b", case=False, na=False),
           "classes_needed_numeric"] = 0
    display(df[[classes_needed_col, "classes_needed_numeric"]].head(10))

,classes_needed_75_pct,classes_needed_numeric
0,4 more,4
1,4 more,4
2,32 more,32
3,✓ Met,0
4,✓ Met,0
5,✓ Met,0
6,✓ Met,0
7,✓ Met,0
8,✓ Met,0
9,✓ Met,0


In [17]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """

# Q15: Create Attendance Flag with apply() and a lambda
df["attendance_flag"] = df[attendance_col].apply(
    lambda value: "Defaulter" if pd.notna(value) and value < 75 else "Safe"
)
display(df[[name_col, attendance_col, "attendance_flag"]].head(10))

,student_name,attendance_pct,attendance_flag
0,ABHIJITH,67.0,Defaulter
1,ABHISHEK KRISHNA P,67.0,Defaulter
2,ADITYA,8.0,Defaulter
3,AKSHATA JAKAREDDY,92.0,Safe
4,ANANYA,83.0,Safe
5,ANANYASHREE,83.0,Safe
6,ANKITA TARANATH MESTA,83.0,Safe
7,ANVITH S SHERIGAR,100.0,Safe
8,APEKSHA,83.0,Safe
9,ARLENE VELORA FERNANDES,75.0,Safe


In [18]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""

# Q16: Bin attendance into the requested ranges
df["attendance_range"] = pd.cut(
    df[attendance_col],
    bins=[0, 50, 75, 90, 100],
    labels=["Critical", "Warning", "Good", "Excellent"],
    include_lowest=True,
    right=True
)
display(df[[name_col, attendance_col, "attendance_range"]].head(10))

,student_name,attendance_pct,attendance_range
0,ABHIJITH,67.0,Warning
1,ABHISHEK KRISHNA P,67.0,Warning
2,ADITYA,8.0,Critical
3,AKSHATA JAKAREDDY,92.0,Excellent
4,ANANYA,83.0,Good
5,ANANYASHREE,83.0,Good
6,ANKITA TARANATH MESTA,83.0,Good
7,ANVITH S SHERIGAR,100.0,Excellent
8,APEKSHA,83.0,Good
9,ARLENE VELORA FERNANDES,75.0,Warning


### Section 5: Advanced Analysis 

In [19]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""

# Q17: Pivot table showing student counts by status and attendance category
pivot_summary = pd.pivot_table(
    df,
    index="status",
    columns="attendance_category",
    values=name_col,
    aggfunc="count",
    fill_value=0,
    observed=False
)
display(pivot_summary)

attendance_category,Poor,Average,Good,Excellent
status,,,,
Critical,20,3,0,0
Safe,0,0,0,13
Warning,0,0,23,0


In [20]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""

# Q18: Find repeated base names (e.g., ANANYA, ANANYA 2)
# Remove trailing numbers/suffixes and normalize case/spacing.
df["base_name"] = (
    df[name_col].astype("string")
    .str.strip()
    .str.replace(r"\s*\(?\d+\)?$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
)
repeated_bases = df["base_name"].value_counts()
repeated_bases = repeated_bases[repeated_bases > 1].index
display(df.loc[df["base_name"].isin(repeated_bases)].sort_values("base_name"))

,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75_pct,last_present_date,consecutive_absences,attendance_category,risk_score,classes_needed_numeric,attendance_flag,attendance_range,base_name


In [21]:
"""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""

# Q19: Create a hypothetical student-details table for the merge exercise.
# The supplied workbook has no Branch/Semester details table. Since the question
# explicitly asks for a hypothetical table, these values are illustrative only.
roll_col = "roll_no" if "roll_no" in df.columns else "roll_number"

student_details = df[[roll_col]].drop_duplicates().copy()
student_details = student_details.rename(columns={roll_col: "roll_no"})
student_details["branch"] = "AI&ML (hypothetical)"
student_details["semester"] = "5th Semester (hypothetical)"

# Inner join: keeps records whose roll numbers exist in both tables.
merged_df = df.merge(
    student_details,
    left_on=roll_col,
    right_on="roll_no",
    how="inner",
    validate="many_to_one"
)
print("Hypothetical Student_Details table:")
display(student_details.head())
print("Merged attendance and student-details data:")
display(merged_df)
print("Merged shape:", merged_df.shape)

Hypothetical Student_Details table:


,roll_no,branch,semester
0,4MW24AI001,AI&ML (hypothetical),5th Semester (hypothetical)
1,4MW24AI002,AI&ML (hypothetical),5th Semester (hypothetical)
2,4MW24AI003,AI&ML (hypothetical),5th Semester (hypothetical)
3,4MW24AI004,AI&ML (hypothetical),5th Semester (hypothetical)
4,4MW24AI005,AI&ML (hypothetical),5th Semester (hypothetical)


Merged attendance and student-details data:


,roll_no,student_name,subject,total_classes,present,absent,excused,attendance_pct,status,classes_needed_75_pct,last_present_date,consecutive_absences,attendance_category,risk_score,classes_needed_numeric,attendance_flag,attendance_range,base_name,branch,semester
0,4MW24AI001,ABHIJITH,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67.0,Critical,4 more,2026-10-09,1,Average,2.2,4,Defaulter,Warning,ABHIJITH,AI&ML (hypothetical),5th Semester (hypothetical)
1,4MW24AI002,ABHISHEK KRISHNA P,Data Science & GenAI - 3nd Yr- AIML,12,8,4,0,67.0,Critical,4 more,2026-10-09,1,Average,2.2,4,Defaulter,Warning,ABHISHEK KRISHNA P,AI&ML (hypothetical),5th Semester (hypothetical)
2,4MW24AI003,ADITYA,Data Science & GenAI - 3nd Yr- AIML,12,1,11,0,8.0,Critical,32 more,2026-09-22,5,Poor,7.4,32,Defaulter,Critical,ADITYA,AI&ML (hypothetical),5th Semester (hypothetical)
3,4MW24AI004,AKSHATA JAKAREDDY,Data Science & GenAI - 3nd Yr- AIML,12,11,1,0,92.0,Safe,✓ Met,2026-10-10,0,Excellent,0.4,0,Safe,Excellent,AKSHATA JAKAREDDY,AI&ML (hypothetical),5th Semester (hypothetical)
4,4MW24AI005,ANANYA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83.0,Warning,✓ Met,2026-10-09,1,Good,1.4,0,Safe,Good,ANANYA,AI&ML (hypothetical),5th Semester (hypothetical)
5,4MW24AI006,ANANYASHREE,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83.0,Warning,✓ Met,2026-10-09,1,Good,1.4,0,Safe,Good,ANANYASHREE,AI&ML (hypothetical),5th Semester (hypothetical)
6,4MW24AI007,ANKITA TARANATH MESTA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83.0,Warning,✓ Met,2026-10-09,1,Good,1.4,0,Safe,Good,ANKITA TARANATH MESTA,AI&ML (hypothetical),5th Semester (hypothetical)
7,4MW24AI008,ANVITH S SHERIGAR,Data Science & GenAI - 3nd Yr- AIML,12,12,0,0,100.0,Safe,✓ Met,2026-10-10,0,Excellent,0.0,0,Safe,Excellent,ANVITH S SHERIGAR,AI&ML (hypothetical),5th Semester (hypothetical)
8,4MW24AI009,APEKSHA,Data Science & GenAI - 3nd Yr- AIML,12,10,2,0,83.0,Warning,✓ Met,2026-10-09,1,Good,1.4,0,Safe,Good,APEKSHA,AI&ML (hypothetical),5th Semester (hypothetical)
9,4MW24AI010,ARLENE VELORA FERNANDES,Data Science & GenAI - 3nd Yr- AIML,12,9,3,0,75.0,Warning,✓ Met,2026-10-09,1,Good,1.8,0,Safe,Warning,ARLENE VELORA FERNANDES,AI&ML (hypothetical),5th Semester (hypothetical)


Merged shape: (59, 20)


In [22]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""

# Q20: Export critical students, status summary, and top 10 risk scores
critical_students = df[df["status"].astype("string").str.strip().str.casefold() == "critical"]
critical_students.to_csv("critical_students.csv", index=False)

summary_by_status = df.groupby("status", dropna=False).agg(
    student_count=(name_col, "size"),
    average_attendance=(attendance_col, "mean"),
    max_consecutive_absences=(consecutive_col, "max")
).reset_index()
summary_by_status.to_csv("attendance_summary_by_status.csv", index=False)

if "risk_score" not in df.columns:
    df["risk_score"] = (
        pd.to_numeric(df["absent"], errors="coerce").fillna(0) * 0.4
        + pd.to_numeric(df[consecutive_col], errors="coerce").fillna(0) * 0.6
    )
df.nlargest(10, "risk_score").to_csv("top_10_risk_students.csv", index=False)

print("Exported: critical_students.csv, attendance_summary_by_status.csv, top_10_risk_students.csv")

Exported: critical_students.csv, attendance_summary_by_status.csv, top_10_risk_students.csv
